# Introduction

In the last lesson, we saw that a convolutional classifier has two parts: a convolutional **base** and a **head** of dense layers. We learned that the job of the base is to extract visual features from an image, which the head would then use to classify the image.

Over the next few lessons, we're going to learn about the two most important types of layers that you'll usually find in the base of a convolutional image classifier. These are the **convolutional layer** with **ReLU activation**, and the **maximum pooling layer**. In Lesson 5, you'll learn how to design your own convnet by composing these layers into blocks that perform the feature extraction.

This lesson is about the convolutional layer with its ReLU activation function.

---

# Feature Extraction

Before we get into the details of convolution, let's discuss the *purpose* of these layers in the network. We're going to see how these three operations (convolution, ReLU, and maximum pooling) are used to implement the feature extraction process.

The **feature extraction** performed by the base consists of **three basic operations**:

1. **Filter** an image for a particular feature (convolution)
2. **Detect** that feature within the filtered image (ReLU)
3. **Condense** the image to enhance the features (maximum pooling)

The next figure illustrates this process. You can see how these three operations are able to isolate some particular characteristic of the original image (in this case, horizontal lines).

```mermaid
flowchart LR
    A[Original Image] --> B[Convolution\nFilter for horizontal lines]
    B --> C[ReLU\nKeep strong activations]
    C --> D[Max Pooling\nCondense and enhance]
    D --> E[Extracted Feature Map\nHorizontal lines emphasized]
```


## Setup

We'll start by importing the core libraries used throughout this lesson.

- `TensorFlow` for convolution, ReLU, and pooling operations
- `NumPy` for array utilities
- `Matplotlib` for visualization

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.utils import image_dataset_from_directory

## Reproducibility and Plot Defaults

As in the previous notebook, we'll fix random seeds for reproducible outputs and configure plotting defaults.

In [ ]:
def set_seed(seed=31415):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


set_seed(31415)

plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large', titleweight='bold', titlesize=16, titlepad=10)
plt.rc('image', cmap='magma')
warnings.filterwarnings('ignore')

## Load a Sample Image

We'll load one image from the `Bus` vs `motorcycle` dataset and use it to visualize how each operation transforms image information.

In [ ]:
candidate_data_dirs = [
    os.path.join('dataset', 'bus_motorcycle'),
    os.path.join('..', 'dataset', 'bus_motorcycle'),
]
data_dir = next((d for d in candidate_data_dirs if os.path.isdir(d)), None)
if data_dir is None:
    raise FileNotFoundError(
        f"Could not find bus_motorcycle dataset. Tried {candidate_data_dirs}. Current directory: {os.getcwd()}"
    )

sample_ds = image_dataset_from_directory(
    data_dir,
    labels='inferred',
    label_mode='int',
    image_size=(128, 128),
    batch_size=1,
    shuffle=True,
    seed=31415,
)

class_names = sample_ds.class_names
image_batch, label_batch = next(iter(sample_ds))
image = image_batch[0]
label = int(label_batch[0].numpy())

plt.figure(figsize=(4, 4))
plt.imshow(tf.cast(image, tf.uint8))
plt.title(f'Original Image ({class_names[label]})')
plt.axis('off')
plt.show()

## Convolution: Filtering for Patterns

We'll apply simple hand-crafted kernels to the grayscale image:

- Horizontal edge detector
- Vertical edge detector

This highlights how convolution responds strongly where matching patterns appear.

In [ ]:
image_f = tf.image.convert_image_dtype(image, dtype=tf.float32)
gray = tf.image.rgb_to_grayscale(image_f)
gray_4d = gray[tf.newaxis, ...]  # (1, H, W, 1)

kernel_h = tf.constant([
    [-1.0, -1.0, -1.0],
    [ 0.0,  0.0,  0.0],
    [ 1.0,  1.0,  1.0],
], dtype=tf.float32)

kernel_v = tf.constant([
    [-1.0, 0.0, 1.0],
    [-1.0, 0.0, 1.0],
    [-1.0, 0.0, 1.0],
], dtype=tf.float32)

kernel_h = kernel_h[:, :, tf.newaxis, tf.newaxis]
kernel_v = kernel_v[:, :, tf.newaxis, tf.newaxis]

conv_h = tf.nn.conv2d(gray_4d, kernel_h, strides=1, padding='SAME')[0, :, :, 0]
conv_v = tf.nn.conv2d(gray_4d, kernel_v, strides=1, padding='SAME')[0, :, :, 0]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(gray[:, :, 0], cmap='gray')
axes[0].set_title('Grayscale Input')
axes[0].axis('off')

axes[1].imshow(conv_h, cmap='magma')
axes[1].set_title('Convolution: Horizontal Edges')
axes[1].axis('off')

axes[2].imshow(conv_v, cmap='magma')
axes[2].set_title('Convolution: Vertical Edges')
axes[2].axis('off')

plt.show()

## ReLU: Detecting Strong Activations

After convolution, negative responses are usually less useful for feature presence. ReLU keeps positive activations and sets negatives to zero:

$$\text{ReLU}(x)=\max(0,x)$$

In [ ]:
relu_h = tf.nn.relu(conv_h)
relu_v = tf.nn.relu(conv_v)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

axes[0, 0].imshow(conv_h, cmap='magma')
axes[0, 0].set_title('Before ReLU (Horizontal)')
axes[0, 0].axis('off')

axes[0, 1].imshow(relu_h, cmap='magma')
axes[0, 1].set_title('After ReLU (Horizontal)')
axes[0, 1].axis('off')

axes[1, 0].imshow(conv_v, cmap='magma')
axes[1, 0].set_title('Before ReLU (Vertical)')
axes[1, 0].axis('off')

axes[1, 1].imshow(relu_v, cmap='magma')
axes[1, 1].set_title('After ReLU (Vertical)')
axes[1, 1].axis('off')

plt.show()

## Max Pooling: Condensing Features

Max pooling downsamples feature maps by keeping the strongest activation in each local region. This makes features more compact and often more robust.

For a 2x2 pool with stride 2, width and height are roughly halved.

In [ ]:
relu_h_4d = relu_h[tf.newaxis, :, :, tf.newaxis]
pooled_h = tf.nn.max_pool2d(relu_h_4d, ksize=2, strides=2, padding='VALID')[0, :, :, 0]

print(f'Feature map shape before pooling: {relu_h.shape}')
print(f'Feature map shape after pooling : {pooled_h.shape}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(relu_h, cmap='magma')
axes[0].set_title('After ReLU')
axes[0].axis('off')

axes[1].imshow(pooled_h, cmap='magma')
axes[1].set_title('After Max Pooling (2x2)')
axes[1].axis('off')

plt.show()

## A Compact Conv Block in Keras

In practice, these operations are stacked as layers:

1. `Conv2D` learns filters automatically
2. `ReLU` detects activations
3. `MaxPooling2D` condenses feature maps

In [ ]:
conv_block = tf.keras.Sequential([
    layers.Input(shape=(128, 128, 3)),
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(filters=16, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPooling2D(pool_size=2),
])

conv_block.summary()

## Key Takeaways

- **Convolution** filters the image to highlight specific patterns.
- **ReLU** keeps strong positive activations and removes weak/negative responses.
- **Max pooling** reduces spatial size while preserving salient features.

Together, these form the core feature-extraction pipeline used in convolutional neural networks.